In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"


import random
import time

import tqdm
import wandb
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
    EmpiricalNormalization2D,
    SimpleReplayBufferGNN,
    save_params,
)

from fast_td3 import Critic
from fast_td3.actors import ActorEGNN

In [ ]:
from fast_td3.hyperparams import HumanoidBenchArgs

args = HumanoidBenchArgs(
    env_name="h1-stand-v0",
    total_timesteps=1000,
    render_interval=10,
    eval_interval=10,
    num_envs=4,
    batch_size=1024,
)

In [ ]:
# NOTE: GPU-Related Configurations

amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

In [ ]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
envs = HumanoidBenchEnv(args.env_name, args.num_envs, device=device)
eval_envs = envs
render_env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

In [ ]:
n_act = envs.num_actions
n_obs = envs.num_obs if type(envs.num_obs) == int else envs.num_obs[0]
if envs.asymmetric_obs:
    n_critic_obs = (
        envs.num_privileged_obs
        if type(envs.num_privileged_obs) == int
        else envs.num_privileged_obs[0]
    )
else:
    n_critic_obs = n_obs
action_low, action_high = -1.0, 1.0

In [ ]:
# NOTE: Initialize Normalizer, Actor, and Critic

if args.obs_normalization:
    obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
    critic_obs_normalizer = EmpiricalNormalization(shape=n_critic_obs, device=device)
    xpos_normalizer = EmpiricalNormalization2D(shape=(23, 3), device=device)
else:
    obs_normalizer = nn.Identity()
    critic_obs_normalizer = nn.Identity()

normalize_obs = obs_normalizer.forward
normalize_critic_obs = critic_obs_normalizer.forward
normalize_xpos = xpos_normalizer.forward

# Actor setup
actor = ActorEGNN(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    init_scale=args.init_scale,
    hidden_dim=args.actor_hidden_dim,
)

# the twin actor
actor_detach = ActorEGNN(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    init_scale=args.init_scale,
    hidden_dim=args.actor_hidden_dim,
)

from_module(actor).data.to_module(actor_detach)
policy = actor_detach.explore

# critic
qnet = Critic(
    n_obs=n_critic_obs,
    n_act=n_act,
    num_atoms=args.num_atoms,
    v_min=args.v_min,
    v_max=args.v_max,
    hidden_dim=args.critic_hidden_dim,
    device=device,
)

qnet_target = Critic(
    n_obs=n_critic_obs,
    n_act=n_act,
    num_atoms=args.num_atoms,
    v_min=args.v_min,
    v_max=args.v_max,
    hidden_dim=args.critic_hidden_dim,
    device=device,
)
qnet_target.load_state_dict(qnet.state_dict())

q_optimizer = optim.AdamW(
    list(qnet.parameters()),
    lr=args.critic_learning_rate,
    weight_decay=args.weight_decay,
)
actor_optimizer = optim.AdamW(
    list(actor.parameters()),
    lr=args.actor_learning_rate,
    weight_decay=args.weight_decay,
)

rb = SimpleReplayBufferGNN(
    n_env=args.num_envs,
    buffer_size=args.buffer_size,
    n_obs=n_obs,
    n_act=n_act,
    n_critic_obs=n_critic_obs,
    asymmetric_obs=envs.asymmetric_obs,
    playground_mode=env_type == "mujoco_playground",
    n_steps=args.num_steps,
    gamma=args.gamma,
    device=device,
)

In [ ]:
obs, xpos = render_env.reset()

from fast_td3.egnn_clean import EGNN

egnn = EGNN(in_node_nf=1, hidden_nf=32, out_node_nf=1, in_edge_nf=0, batch_size=args.batch_size, device=device)
h, x, edges, edge_attr = egnn.build_batched_egnn_input(obs, xpos)

print(f"Shape of h: {h.shape}")
print(f"Shape of x: {x.shape}")
print(len(edges))         
print(f"Shape of edges: {edges[0].shape}")


output = egnn.forward(h=h,x=x,edges=edges,edge_attr=edge_attr)

print(output.shape)

actor(obs, xpos)